In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from arch import arch_model
import warnings
warnings.filterwarnings("ignore")

In [ ]:
pip install arch

In [ ]:
DATA_PATH = "C:/Users/your_name/Downloads/" ###### change this to the folder where you saved the files

returns = pd.read_csv(DATA_PATH + "returns.csv", index_col=0, parse_dates=True)
raw     = pd.read_csv(DATA_PATH + "prices.csv",  index_col=0, parse_dates=True)
spy_ret = returns["SPY"]

print(f"Period: {returns.index[0].date()} to {returns.index[-1].date()}")
print(f"Total trading days: {len(returns)}")
print(returns["Regime"].value_counts().reindex([
    "Pre-GFC","GFC","Recovery","Low-Vol Bull",
    "COVID","Post-COVID","2022 Bear",
    "2023 Recovery","2024 AI Rally","2025-2026"
]))

In [ ]:
#####  VaR Models #####

WINDOW = 250
Z_01   = stats.norm.ppf(0.01)

def parametric_var(r, window=WINDOW):
    mu  = r.rolling(window).mean()
    sig = r.rolling(window).std()
    return -(mu + Z_01 * sig)

def hs_var(r, window=WINDOW):
    return -r.rolling(window).quantile(0.01)

def fhs_var(r, window=WINDOW):
    data = r.dropna().values * 100
    n    = len(data)
    out  = np.full(n, np.nan)

    for t in range(window, n):
        try:
            w       = data[t - window:t]
            res     = arch_model(w, vol="Garch", p=1, q=1,
                                 dist="normal", rescale=False).fit(disp="off")
            sig_t1  = np.sqrt(res.forecast(horizon=1).variance.values[-1, 0])
            q_resid = np.percentile(res.std_resid, 1)
            out[t]  = -(sig_t1 * q_resid) / 100
        except Exception:
            out[t] = np.nan

        if t % 500 == 0:
            print(f"  progress: {t}/{n} ({t/n*100:.1f}%)")

    return pd.Series(out, index=r.index)

def hs_es(r, window=WINDOW, conf=0.975):
    out = np.full(len(r), np.nan)
    for t in range(window, len(r)):
        w      = r.iloc[t - window:t]
        cutoff = w.quantile(1 - conf)
        tail   = w[w <= cutoff]
        out[t] = -tail.mean() if len(tail) > 0 else np.nan
    return pd.Series(out, index=r.index)

print("Parametric VaR...")
pvar = parametric_var(spy_ret)

print("HS VaR...")
hsvar = hs_var(spy_ret)

print("FHS VaR (10-15 min)...")
fhsvar = fhs_var(spy_ret)

print("ES 97.5%...")
es_hs = hs_es(spy_ret)

print("Done.")

In [ ]:
##### Backtesting #####

results = pd.DataFrame({
    "Return":    spy_ret,
    "Param_VaR": pvar,
    "HS_VaR":    hsvar,
    "FHS_VaR":   fhsvar,
    "ES_975":    es_hs,
    "Regime":    returns["Regime"]
}).dropna(subset=["Param_VaR","HS_VaR"])

results["Ex_Param"] = (results["Return"] < -results["Param_VaR"]).astype(int)
results["Ex_HS"]    = (results["Return"] < -results["HS_VaR"]).astype(int)
results["Ex_FHS"]   = (results["Return"] < -results["FHS_VaR"]).astype(int)

def traffic_light(n):
    if n <= 4:   return "Green"
    elif n <= 9: return "Yellow"
    else:        return "Red"

def kupiec_test(exceptions, n, p=0.01):
    x = exceptions.sum()
    if x == 0 or x == n:
        return np.nan, np.nan
    p_hat = x / n
    lr    = -2 * (x * np.log(p / p_hat) + (n - x) * np.log((1 - p) / (1 - p_hat)))
    return round(lr, 4), round(1 - stats.chi2.cdf(lr, df=1), 4)

def christoffersen_test(exceptions):
    ex = exceptions.values
    n00 = n01 = n10 = n11 = 0
    for t in range(1, len(ex)):
        if   ex[t-1]==0 and ex[t]==0: n00+=1
        elif ex[t-1]==0 and ex[t]==1: n01+=1
        elif ex[t-1]==1 and ex[t]==0: n10+=1
        elif ex[t-1]==1 and ex[t]==1: n11+=1

    p01 = n01/(n00+n01) if (n00+n01)>0 else 0
    p11 = n11/(n10+n11) if (n10+n11)>0 else 0
    p   = (n01+n11)/(n00+n01+n10+n11)

    if p==0 or p==1 or p01==0 or p01==1 or p11==0 or p11==1:
        return np.nan, np.nan

    lr = -2*(
        (n00+n10)*np.log(1-p) + (n01+n11)*np.log(p) -
        n00*np.log(1-p01) - n01*np.log(p01) -
        n10*np.log(1-p11) - n11*np.log(p11)
    )
    return round(lr, 4), round(1 - stats.chi2.cdf(lr, df=1), 4)

regime_order = [
    "Pre-GFC","GFC","Recovery","Low-Vol Bull",
    "COVID","Post-COVID","2022 Bear",
    "2023 Recovery","2024 AI Rally","2025-2026"
]

print(f"{'Regime':<18} {'N':>5} | {'Param':>5} {'Zone':>8} | {'HS':>5} {'Zone':>8} | {'FHS':>5} {'Zone':>8}")
print("-" * 72)

for regime in regime_order:
    sub = results[results["Regime"]==regime]
    if len(sub)==0: continue
    p_ex   = sub["Ex_Param"].sum()
    hs_ex  = sub["Ex_HS"].sum()
    fhs_ex = sub["Ex_FHS"].sum()
    print(f"{regime:<18} {len(sub):>5} | {p_ex:>5} {traffic_light(p_ex):>8} | {hs_ex:>5} {traffic_light(hs_ex):>8} | {fhs_ex:>5} {traffic_light(fhs_ex):>8}")

print("\n")
print(f"{'Model':<12} {'Exceptions':>10} {'Rate':>8} {'Kupiec_p':>10} {'CC_p':>10} {'Reject':>8}")
print("-" * 60)

for col, name in [("Ex_Param","Parametric"),("Ex_HS","HS"),("Ex_FHS","FHS")]:
    ex       = results[col]
    x        = ex.sum()
    _, kup_p = kupiec_test(ex, len(ex))
    _, cc_p  = christoffersen_test(ex)
    reject   = "Yes" if kup_p < 0.05 else "No"
    print(f"{name:<12} {x:>10} {x/len(ex):>8.4f} {kup_p:>10} {cc_p:>10} {reject:>8}")

In [ ]:
##### Save #####

results.to_csv(DATA_PATH + "var_results.csv")
print(f"Saved to {DATA_PATH}var_results.csv")